# Left Myotome Backbone and Continuous Gene Trends

Reconstruct the left myotome backbone and associate continuous anterior-posterior position with gene expression.

This curated notebook targets the current Dynamo-free Spateo API. Edit the path/configuration cells for a new system before execution.


In [ ]:
import sys
from pathlib import Path
import numpy as np

import spateo as st


In [ ]:
cpo1 = [
    (7235.672822135923, -9333.743725966886, 29537.530999873827),
    (5265.3827, 317.52565000000004, 1200.0),
    (-0.9910735612049205, -0.13108990476291582, 0.024261763123203748),
]

cpo2 = [
    (-1508.524414183453, -10974.135117384183, 25563.352608545407),
    (4720.1671, 112.69174999999996, 1200.0),
    (-0.6800775362520292, 0.7171120242306434, 0.15246274754575018),
]

cpo3 = [
    (-134.8569326485603, -22618.34076227472, 15863.43005107714),
    (4720.1671, 112.69174999999996, 1200.0),
    (-0.9734993446053986, 0.06684381746631253, -0.21870283518827516),
]

cpo4 = [
    (-5006.789177572386, -25557.216094653202, -115.0783686672523),
    (4720.1671, 112.69174999999996, 1200.0),
    (-0.9334155956236307, 0.35598914126682585, -0.04480019136890938),
]


## Load and validate data


In [ ]:
Myotome_left = st.read_h5ad("/DATA/User/gaomohan/Myotome_v1_left.h5ad")
Myotome_left


## Construct the point-cloud model


In [ ]:
# Reconstruct point cloud model
Myotome_left_pc, _ = st.tdr.construct_pc(
    adata=Myotome_left.copy(), spatial_key="spatial", groupby="celltype"
)

st.tdr.add_model_labels(
    model=Myotome_left_pc,
    labels=["Point Cloud"] * Myotome_left_pc.n_points,
    key_added="backbone",
    where="point_data",
    inplace=True,
    alphamap=0.4,
    colormap="gainsboro",
)


## Reconstruct the surface mesh


In [ ]:
Myotome_left_mesh, _, _ = st.tdr.construct_surface(
    pc=Myotome_left_pc,
    alpha=0.3,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 0.95},
    smooth=4000,
    scale_factor=1.0,
)

st.tdr.add_model_labels(
    model=Myotome_left_mesh,
    labels=["Mesh"] * Myotome_left_mesh.n_cells,
    key_added="backbone",
    where="cell_data",
    inplace=True,
    alphamap=0.3,
    colormap="gainsboro",
)


## Reconstruct the voxel model


In [ ]:
# Reconstruct voxel model
Myotome_left_voxel, _ = st.tdr.voxelize_mesh(
    mesh=Myotome_left_mesh,
    voxel_pc=None,
    key_added="backbone",
    label="Voxel",
    color="gainsboro",
    smooth=300,
)


In [ ]:
st.pl.three_d_multi_plot(
    model=st.tdr.collect_models(
        [st.tdr.collect_models([Myotome_left_pc, Myotome_left_mesh]), Myotome_left_voxel]
    ),
    key="backbone",
    model_style=[["points", "surface"], "surface"],
    jupyter="static",
    cpo=[cpo1],
    shape=(1, 2),
    ambient=[0.2, 0.1],
)


In [ ]:
import torch

from spateo.tdr.models.models_backbone.backbone_methods import NLPCA

def fixed_project(self, data):
    num_dim = data.shape[1]

    data_tensor = torch.tensor(
        data,
        dtype=torch.float32,
    )

    self.model.eval()

    with torch.no_grad():
        output = self.model(data_tensor)

        pts = output.detach().cpu().numpy()

        proj = self.model.intermediate_layer_model.detach().cpu().numpy()

    self.fit_points = pts

    all_data = np.concatenate(
        [pts, proj],
        axis=1,
    )

    all_sorted = all_data[all_data[:, num_dim].argsort()]

    return proj, all_sorted

NLPCA.project = fixed_project


## Reconstruct the anatomical backbone


In [ ]:
backbone, backbone_length, cmap = st.tdr.construct_backbone(
    model=Myotome_left_voxel,
    rd_method="PrinCurve",
    num_nodes=50,
    epochs=400,
    scale_factor=7250,
    color="orangered",
)
st.pl.three_d_plot(
    model=st.tdr.collect_models([Myotome_left_pc, Myotome_left_mesh, backbone]),
    key=["backbone", "backbone", "backbone"],
    opacity=0.2,
    model_style=["points", "surface", "wireframe"],
    model_size=[0, 2, 5],
    show_legend=True,
    jupyter="static",
    cpo=cpo1,
)


In [ ]:
backbone_elpi, backbone_length_elpi, _ = st.tdr.construct_backbone(
    model=Myotome_left_voxel,
    rd_method="ElPiGraph",
    num_nodes=50,
    topology="curve",
    color="orangered",
)


In [ ]:
backbone, backbone_length, _ = st.tdr.construct_backbone(
    model=Myotome_left_voxel,
    rd_method="ElPiGraph",
    num_nodes=30,
    topology="curve",
    Lambda=0.01,
    Mu=0.2,
    color="orangered",
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([Myotome_left_pc, Myotome_left_mesh, backbone_elpi]),
    key=["backbone", "backbone", "backbone"],
    opacity=0.2,
    model_style=["points", "surface", "wireframe"],
    model_size=[0, 2, 5],
    show_legend=True,
    jupyter="static",
    cpo=cpo1,
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([Myotome_left_pc, Myotome_left_mesh, backbone]),
    key=["backbone", "backbone", "backbone"],
    opacity=0.2,
    model_style=["points", "surface", "wireframe"],
    model_size=[0, 2, 5],
    show_legend=True,
    jupyter="static",
    cpo=cpo1,
)


In [ ]:
X = np.asarray(Myotome_left_voxel.points)
B = np.asarray(backbone.points)

print("=== voxel ===")
print("shape:", X.shape)
print("min:", X.min(axis=0))
print("max:", X.max(axis=0))
print("range:", np.ptp(X, axis=0))

print("\n=== backbone ===")
print("n_points:", backbone.n_points)
print("n_cells:", backbone.n_cells)
print("unique points:", np.unique(B, axis=0).shape[0])
print("min:", B.min(axis=0))
print("max:", B.max(axis=0))
print("range:", np.ptp(B, axis=0))

print("\nbackbone_length:", backbone_length)
print("finite:", np.all(np.isfinite(B)))


In [ ]:
from scipy.spatial import KDTree


In [ ]:
backbone_nodes = np.asarray(backbone.points)
backbone_nodes_kdtree = KDTree(np.asarray(backbone_nodes))
_, ii = backbone_nodes_kdtree.query(np.asarray(Myotome_left.obsm["spatial"]), k=1)
Myotome_left.obs["backbone"] = ii


## Associate gene expression with morphology


In [ ]:
st.tl.glm_degs(
    adata=Myotome_left,
    fullModelFormulaStr="~cr(backbone, df=3)",
    key_added="glm_degs",
    qval_threshold=0.05,
    llf_threshold=-1000,
)
print(Myotome_left.uns["glm_degs"]["glm_result"])


## Continuous A–P backbone-gene analysis

This section continues the original left Myotome backbone workflow with a continuous A→P projection. The 30-node ElPiGraph curve is ordered from the visually upper endpoint (A) to the lower endpoint (P) under the notebook's original `cpo1` camera. Every cell is projected to the complete polyline, and Spateo `glm_degs` tests the continuous position using `~cr(backbone_s, df=3)`.

Genes must be detected in at least 3% of cells before testing. Heatmap profiles are Gaussian-kernel smoothed over 200 A–P positions (`bandwidth=0.04`). In the representative-gene panels, point color encodes A–P position only and expression is encoded by y.

- Input: `/DATA/User/gaomohan/Myotome_v1_left.h5ad`
- Output: `/DATA/User/gaomohan/figures/figure3/d/Myotome_backbone_gene_analysis_20260821/left`


In [ ]:
import subprocess

analysis_root = Path(
    "/DATA/User/gaomohan/figures/figure3/d/Myotome_backbone_gene_analysis_20260821"
)
side_output = analysis_root / "left"
analysis_script = analysis_root / "code" / "myotome_backbone_gene_analysis.py"

cmd = [
    sys.executable,
    str(analysis_script),
    "--input-h5ad",
    "/DATA/User/gaomohan/Myotome_v1_left.h5ad",
    "--output-dir",
    str(side_output),
    "--side",
    "left",
    "--min-expressed-fraction",
    "0.03",
    "--reuse-glm-table",
]
# Reconstructs the backbone and reuses the validated GLM table when available.
subprocess.run(cmd, check=True)


In [ ]:
from IPython.display import Image, display

figure_dir = Path(
    "/DATA/User/gaomohan/figures/figure3/d/Myotome_backbone_gene_analysis_20260821/left/figures"
)
for filename in [
    "myotome_left_backbone_orientation_QC.png",
    "myotome_left_continuous_AP_heatmap.png",
    "myotome_left_continuous_AP_composite.png",
]:
    print(filename)
    display(Image(filename=str(figure_dir / filename)))
